# Simulation 1: Mathematische Konvergenz paralleler Deliberation

**Ziel:** Konvergiert iterative parallele Verarbeitung in orthogonalen Unterräumen zu einem stabilen Ergebnis?

Zwei Paradigmen im Vergleich:
- **Fixpunkt-Iteration**: Wiederholung bis Delta < epsilon
- **Energieminimierung** (EBT-inspiriert): Gradient Descent auf gemeinsame Energiefunktion

**Unique Contribution:** Signalverarbeitungs-Metriken (SNR, Phasenkohaerenz, Crest Factor) als Konvergenz-Diagnostik — in keinem der 75+ recherchierten Papers.

Autoren: Toby Brummer & Claude | Datum: 2026-03-29

In [ ]:
# 0. Setup & Imports
import sys
sys.path.insert(0, '/workspace')

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import svd, orth
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Our modules
from sim1_helpers import (
    generate_orthogonal_subspaces,
    load_semantic_vectors,
    merge_average, merge_phase_alignment, merge_frequency_selective, merge_sidechain,
    MERGE_STRATEGIES,
    iterate_fixpoint, iterate_energy,
    apply_diversity_repulsion,
)
from sim1_metrics import (
    compute_snr, compute_phase_coherence, compute_crest_factor,
    compute_all_metrics, halting_decision, compare_halting_methods,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="whitegrid", palette="muted")
print("Setup complete.")

## Experiment A: Basis-Konvergenz

**Kernfrage:** Konvergiert das System überhaupt?

Sweep über Unterraumanzahl × Dimensionalität × Mechanismus bei exakter Orthogonalität.
Wenn das hier scheitert: fundamentales Problem, alles Weitere ist fragwürdig.

In [ ]:
# Experiment A: Basis-Konvergenz (Fixpunkt vs. Energie)
# Exakt orthogonale Unterräume, zufällige Startvektoren, merge_average

configs_a = {
    "n_subspaces": [2, 3, 5, 10],
    "dim": [64, 256, 1024],
}

results_a = []

for n_sub, dim in product(configs_a["n_subspaces"], configs_a["dim"]):
    # subspace_dim muss so gewählt sein, dass n_sub * subspace_dim <= dim
    subspace_dim = dim // (n_sub * 2)
    if subspace_dim < 2:
        continue
    
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    
    subspaces = generate_orthogonal_subspaces(n_sub, dim, subspace_dim)
    workers = [P @ np.random.randn(subspace_dim) for P in subspaces]
    
    for mechanism in ["fixpoint", "energy"]:
        if mechanism == "fixpoint":
            history = iterate_fixpoint(workers, subspaces, merge_average, max_iter=50)
        else:
            history = iterate_energy(workers, subspaces, merge_average, max_iter=50, lr=0.1)
        
        metrics = compute_all_metrics(history)
        
        results_a.append({
            "n_subspaces": n_sub,
            "dim": dim,
            "subspace_dim": subspace_dim,
            "mechanism": mechanism,
            "converged": history["converged"],
            "iterations": history["n_iterations"],
            "final_snr": metrics["snr"][-1] if metrics["snr"] else None,
            "snr_curve": metrics["snr"],
            "delta_curve": history["deltas"],
        })

# Zusammenfassung
print(f"{'n_sub':>5} {'dim':>5} {'mechanism':>10} {'converged':>10} {'iters':>6} {'final_snr':>10}")
print("-" * 55)
for r in results_a:
    snr_str = f"{r['final_snr']:.1f}" if r['final_snr'] and r['final_snr'] != float('inf') else "inf"
    print(f"{r['n_subspaces']:>5} {r['dim']:>5} {r['mechanism']:>10} {str(r['converged']):>10} {r['iterations']:>6} {snr_str:>10}")

In [ ]:
# Visualisierung Experiment A: SNR-Kurven und Konvergenz

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Experiment A: Basis-Konvergenz (exakt orthogonal, merge_average)", fontsize=14)

for col, dim in enumerate(configs_a["dim"]):
    for row, mechanism in enumerate(["fixpoint", "energy"]):
        ax = axes[row, col]
        matching = [r for r in results_a if r["dim"] == dim and r["mechanism"] == mechanism]
        
        for r in matching:
            label = f"n={r['n_subspaces']}"
            snr = r["snr_curve"]
            if snr and all(s != float('inf') and s != float('-inf') for s in snr):
                ax.plot(snr, label=label, marker='o', markersize=3)
        
        ax.set_title(f"{mechanism}, dim={dim}")
        ax.set_xlabel("Iteration")
        ax.set_ylabel("SNR (dB)")
        ax.legend(fontsize=8)
        ax.set_ylim(bottom=-5)

plt.tight_layout()
plt.savefig("/workspace/exp_a_convergence.png", dpi=150)
plt.show()
print("Saved: exp_a_convergence.png")

## Experiment B: Orthogonalitätsgrad-Sweep

**Kernfrage:** Ab welchem Winkel bricht die Konvergenz?

Fixe Konfiguration (dim=256, n=5), Sweep über Orthogonalitätsgrad 0.0 bis 1.0.

In [ ]:
# Experiment B: Orthogonalitäts-Sweep
dim_b = 256
n_sub_b = 5
subspace_dim_b = dim_b // (n_sub_b * 2)  # = 25

orthogonality_levels = [1.0, 0.95, 0.9, 0.8, 0.7, 0.5, 0.3, 0.0]

results_b = []

for ortho in orthogonality_levels:
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    
    subspaces = generate_orthogonal_subspaces(n_sub_b, dim_b, subspace_dim_b, orthogonality=ortho)
    workers = [P @ np.random.randn(subspace_dim_b) for P in subspaces]
    
    for mechanism in ["fixpoint", "energy"]:
        if mechanism == "fixpoint":
            history = iterate_fixpoint(workers, subspaces, merge_average, max_iter=50)
        else:
            history = iterate_energy(workers, subspaces, merge_average, max_iter=50, lr=0.1)
        
        metrics = compute_all_metrics(history)
        
        results_b.append({
            "orthogonality": ortho,
            "mechanism": mechanism,
            "converged": history["converged"],
            "iterations": history["n_iterations"],
            "final_snr": metrics["snr"][-1] if metrics["snr"] else None,
            "snr_curve": metrics["snr"],
        })

# Visualisierung
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Experiment B: Orthogonalitäts-Sweep (dim={dim_b}, n={n_sub_b})", fontsize=14)

for mechanism, ax in [("fixpoint", ax1), ("energy", ax2)]:
    matching = [r for r in results_b if r["mechanism"] == mechanism]
    orthos = [r["orthogonality"] for r in matching]
    iters = [r["iterations"] for r in matching]
    converged = [r["converged"] for r in matching]
    
    colors = ['green' if c else 'red' for c in converged]
    ax.bar(range(len(orthos)), iters, color=colors, alpha=0.7)
    ax.set_xticks(range(len(orthos)))
    ax.set_xticklabels([f"{o:.2f}" for o in orthos], rotation=45)
    ax.set_xlabel("Orthogonalitätsgrad")
    ax.set_ylabel("Iterationen")
    ax.set_title(f"{mechanism} (grün=konvergiert, rot=nicht)")
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='max_iter')

plt.tight_layout()
plt.savefig("/workspace/exp_b_orthogonality.png", dpi=150)
plt.show()
print("Saved: exp_b_orthogonality.png")

## Experiment C: Merge-Strategien-Vergleich

**Kernfrage:** Welcher Merge erhält am meisten Information?

Alle vier Merge-Strategien bei drei Orthogonalitätsgraden (1.0, 0.8, 0.5).

In [ ]:
# Experiment C: Merge-Strategien-Vergleich
dim_c = 256
n_sub_c = 5
subspace_dim_c = dim_c // (n_sub_c * 2)
ortho_levels_c = [1.0, 0.8, 0.5]

results_c = []

for ortho in ortho_levels_c:
    for merge_name, merge_fn in MERGE_STRATEGIES.items():
        for mechanism in ["fixpoint", "energy"]:
            np.random.seed(SEED)
            torch.manual_seed(SEED)
            
            subspaces = generate_orthogonal_subspaces(n_sub_c, dim_c, subspace_dim_c, orthogonality=ortho)
            workers = [P @ np.random.randn(subspace_dim_c) for P in subspaces]
            
            if mechanism == "fixpoint":
                history = iterate_fixpoint(workers, subspaces, merge_fn, max_iter=50)
            else:
                history = iterate_energy(workers, subspaces, merge_fn, max_iter=50, lr=0.1)
            
            metrics = compute_all_metrics(history)
            
            results_c.append({
                "orthogonality": ortho,
                "merge": merge_name,
                "mechanism": mechanism,
                "converged": history["converged"],
                "iterations": history["n_iterations"],
                "final_snr": metrics["snr"][-1] if metrics["snr"] else None,
            })

# Heatmap: Merge x Orthogonalität, Farbe = Iterationen
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Experiment C: Merge-Strategien × Orthogonalität", fontsize=14)

for mechanism, ax in [("fixpoint", ax1), ("energy", ax2)]:
    matching = [r for r in results_c if r["mechanism"] == mechanism]
    
    merges = list(MERGE_STRATEGIES.keys())
    data = np.zeros((len(merges), len(ortho_levels_c)))
    annot = np.empty_like(data, dtype=object)
    
    for r in matching:
        i = merges.index(r["merge"])
        j = ortho_levels_c.index(r["orthogonality"])
        data[i, j] = r["iterations"]
        annot[i, j] = f"{r['iterations']}\n{'✓' if r['converged'] else '✗'}"
    
    sns.heatmap(data, ax=ax, annot=annot, fmt='', cmap='YlOrRd_r',
                xticklabels=[f"ortho={o}" for o in ortho_levels_c],
                yticklabels=merges)
    ax.set_title(f"{mechanism}")

plt.tight_layout()
plt.savefig("/workspace/exp_c_merge_strategies.png", dpi=150)
plt.show()
print("Saved: exp_c_merge_strategies.png")

## Experiment D: Signalmetriken als Diagnostik

**DER zentrale Test für den unique Beitrag.**

Frage: Tracken SNR, Phasenkohaerenz und Crest Factor die Konvergenzqualität besser als reines Delta? Stoppt Signal-basiertes Halting früher bei gleicher oder besserer Qualität?

In [ ]:
# Experiment D: Signalmetriken als Diagnostik
# Detaillierte Analyse der besten Konfigurationen aus A-C

dim_d = 256
n_sub_d = 5
subspace_dim_d = dim_d // (n_sub_d * 2)

# Drei Szenarien: saubere Konvergenz, schwierige Konvergenz, Divergenz
scenarios = [
    {"name": "Clean (ortho=1.0)", "ortho": 1.0},
    {"name": "Noisy (ortho=0.7)", "ortho": 0.7},
    {"name": "Broken (ortho=0.3)", "ortho": 0.3},
]

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
fig.suptitle("Experiment D: Signalmetriken über Iterationen", fontsize=14)

col_labels = ["SNR (dB)", "Phase Coherence (top 3)", "Crest Factor", "Delta"]

for row, scenario in enumerate(scenarios):
    np.random.seed(SEED)
    subspaces = generate_orthogonal_subspaces(n_sub_d, dim_d, subspace_dim_d, orthogonality=scenario["ortho"])
    workers = [P @ np.random.randn(subspace_dim_d) for P in subspaces]
    
    history = iterate_fixpoint(workers, subspaces, merge_average, max_iter=50)
    metrics = compute_all_metrics(history)
    
    # Halting-Vergleich
    comparison = compare_halting_methods(history, metrics, delta_epsilon=1e-4)
    
    # SNR
    ax = axes[row, 0]
    snr_vals = [s for s in metrics["snr"] if s != float('inf') and s != float('-inf')]
    ax.plot(snr_vals, 'b-o', markersize=3)
    if comparison.get("signal_stop", len(snr_vals)) < len(snr_vals):
        ax.axvline(x=comparison["signal_stop"], color='red', linestyle='--', label=f'Signal: {comparison["signal_decision"]}')
    if comparison.get("delta_stop", len(snr_vals)) < len(snr_vals):
        ax.axvline(x=comparison["delta_stop"], color='green', linestyle='--', label=f'Delta stop')
    ax.set_ylabel(scenario["name"])
    ax.legend(fontsize=7)
    if row == 0:
        ax.set_title(col_labels[0])
    
    # Phase Coherence (top 3 components)
    ax = axes[row, 1]
    coh_matrix = np.array([c[:3] for c in metrics["coherence"]])
    for k in range(min(3, coh_matrix.shape[1])):
        ax.plot(coh_matrix[:, k], marker='o', markersize=3, label=f'Comp {k+1}')
    ax.set_ylim(-0.1, 1.1)
    ax.legend(fontsize=7)
    if row == 0:
        ax.set_title(col_labels[1])
    
    # Crest Factor
    ax = axes[row, 2]
    ax.plot(metrics["crest_factor"], 'r-o', markersize=3)
    ax.axhline(y=3.0, color='gray', linestyle=':', alpha=0.5, label='Threshold')
    ax.legend(fontsize=7)
    if row == 0:
        ax.set_title(col_labels[2])
    
    # Delta
    ax = axes[row, 3]
    ax.plot(history["deltas"], 'g-o', markersize=3)
    ax.set_yscale('log')
    ax.axhline(y=1e-4, color='gray', linestyle=':', alpha=0.5, label='epsilon')
    ax.legend(fontsize=7)
    if row == 0:
        ax.set_title(col_labels[3])
    
    # Print halting comparison
    print(f"{scenario['name']}:")
    print(f"  Delta stops at iter {comparison.get('delta_stop', 'never')}")
    print(f"  Signal stops at iter {comparison.get('signal_stop', 'never')} ({comparison.get('signal_decision', 'n/a')})")
    if 'snr_at_delta_stop' in comparison:
        print(f"  SNR when delta stops: {comparison['snr_at_delta_stop']:.1f} dB")
    if 'delta_at_signal_stop' in comparison:
        print(f"  Delta when signal stops: {comparison['delta_at_signal_stop']:.6f}")
    print()

plt.tight_layout()
plt.savefig("/workspace/exp_d_signal_metrics.png", dpi=150)
plt.show()
print("Saved: exp_d_signal_metrics.png")

## Experiment E: Diversity-Enforcement

**Kernfrage:** Kollabieren Worker ohne Repulsion?

Vergleich: diversity_strength = 0, 0.01, 0.05, 0.1, 0.5. Kollaps-Metrik: mittlere paarweise Cosine Similarity > 0.95 = Kollaps.

In [ ]:
# Experiment E: Diversity-Enforcement

def mean_pairwise_cosine(states):
    """Mittlere paarweise Cosine Similarity. >0.95 = Kollaps."""
    flat = [s.flatten() for s in states]
    sims = []
    for i in range(len(flat)):
        for j in range(i + 1, len(flat)):
            cos = np.dot(flat[i], flat[j]) / (np.linalg.norm(flat[i]) * np.linalg.norm(flat[j]) + 1e-8)
            sims.append(cos)
    return np.mean(sims) if sims else 0.0

dim_e = 256
n_sub_e = 5
subspace_dim_e = dim_e // (n_sub_e * 2)
diversity_levels = [0.0, 0.01, 0.05, 0.1, 0.5]

fig, axes = plt.subplots(2, len(diversity_levels), figsize=(20, 8))
fig.suptitle("Experiment E: Diversity-Enforcement", fontsize=14)

for col, div_strength in enumerate(diversity_levels):
    for row, mechanism in enumerate(["fixpoint", "energy"]):
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        
        subspaces = generate_orthogonal_subspaces(n_sub_e, dim_e, subspace_dim_e)
        workers = [P @ np.random.randn(subspace_dim_e) for P in subspaces]
        
        if mechanism == "fixpoint":
            history = iterate_fixpoint(workers, subspaces, merge_average,
                                       max_iter=50, diversity_strength=div_strength)
        else:
            history = iterate_energy(workers, subspaces, merge_average,
                                     max_iter=50, lr=0.1, diversity_strength=div_strength)
        
        # Cosine Similarity über Iterationen
        cos_per_iter = [mean_pairwise_cosine(states) for states in history["states"]]
        
        ax = axes[row, col]
        ax.plot(cos_per_iter, 'purple', marker='o', markersize=3)
        ax.axhline(y=0.95, color='red', linestyle='--', alpha=0.5, label='Collapse threshold')
        ax.set_ylim(-0.2, 1.1)
        ax.set_xlabel("Iteration")
        if col == 0:
            ax.set_ylabel(f"{mechanism}\nCosine Sim")
        ax.set_title(f"div={div_strength}" if row == 0 else "")
        ax.legend(fontsize=7)
        
        final_cos = cos_per_iter[-1] if cos_per_iter else 0
        status = "COLLAPSE" if final_cos > 0.95 else "OK"
        print(f"{mechanism}, div={div_strength}: final_cosine={final_cos:.3f} [{status}], "
              f"converged={history['converged']}, iters={history['n_iterations']}")

plt.tight_layout()
plt.savefig("/workspace/exp_e_diversity.png", dpi=150)
plt.show()
print("\nSaved: exp_e_diversity.png")

## Experiment F: Semantische vs. zufällige Vektoren

**Kernfrage:** Verhält sich realistisch strukturierter Input anders als Gaussian noise?

In [ ]:
# Experiment F: Semantische vs. zufällige Vektoren

dim_f = 256
n_sub_f = 5
subspace_dim_f = dim_f // (n_sub_f * 2)

results_f = {}

for vec_type in ["random", "semantic"]:
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    
    subspaces = generate_orthogonal_subspaces(n_sub_f, dim_f, subspace_dim_f)
    
    if vec_type == "random":
        workers = [P @ np.random.randn(subspace_dim_f) for P in subspaces]
    else:
        # Semantische Vektoren in die Unterräume projizieren
        sem_vecs = load_semantic_vectors(n_sub_f, dim_f)
        workers = [P @ (P.T @ sem_vecs[i]) for i, P in enumerate(subspaces)]
    
    history = iterate_fixpoint(workers, subspaces, merge_average, max_iter=50)
    metrics = compute_all_metrics(history)
    results_f[vec_type] = {"history": history, "metrics": metrics}

# Vergleichsplot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Experiment F: Random vs. Semantische Vektoren (Fixpunkt)", fontsize=14)

for vec_type, style in [("random", "b-o"), ("semantic", "r-s")]:
    m = results_f[vec_type]["metrics"]
    h = results_f[vec_type]["history"]
    
    # SNR
    snr = [s for s in m["snr"] if s != float('inf') and s != float('-inf')]
    axes[0].plot(snr, style, markersize=3, label=vec_type)
    
    # Crest Factor
    axes[1].plot(m["crest_factor"], style, markersize=3, label=vec_type)
    
    # Delta
    axes[2].plot(h["deltas"], style, markersize=3, label=vec_type)

axes[0].set_title("SNR (dB)")
axes[1].set_title("Crest Factor")
axes[2].set_title("Delta (log)")
axes[2].set_yscale('log')

for ax in axes:
    ax.legend()
    ax.set_xlabel("Iteration")

plt.tight_layout()
plt.savefig("/workspace/exp_f_semantic.png", dpi=150)
plt.show()

for vt in ["random", "semantic"]:
    h = results_f[vt]["history"]
    m = results_f[vt]["metrics"]
    print(f"{vt}: converged={h['converged']}, iters={h['n_iterations']}, "
          f"final_snr={m['snr'][-1]:.1f} dB")

print("\nSaved: exp_f_semantic.png")

## Zusammenfassung & Entscheidung

In [ ]:
# Zusammenfassung: Automatisierte Entscheidungsmatrix

# Aus Experiment A: Konvergiert es?
convergence_works = any(r["converged"] for r in results_a)
best_mechanism = "fixpoint"  # default
fp_iters = [r["iterations"] for r in results_a if r["mechanism"] == "fixpoint" and r["converged"]]
en_iters = [r["iterations"] for r in results_a if r["mechanism"] == "energy" and r["converged"]]
if en_iters and (not fp_iters or np.mean(en_iters) < np.mean(fp_iters)):
    best_mechanism = "energy"

# Aus Experiment C: Bester Merge
best_merge = "average"
merge_scores = {}
for r in results_c:
    if r["converged"] and r["mechanism"] == best_mechanism:
        key = r["merge"]
        if key not in merge_scores:
            merge_scores[key] = []
        merge_scores[key].append(r["iterations"])
if merge_scores:
    best_merge = min(merge_scores, key=lambda k: np.mean(merge_scores[k]))

# Aus Experiment F: Semantischer Unterschied?
semantic_diff = abs(
    results_f["random"]["history"]["n_iterations"] - 
    results_f["semantic"]["history"]["n_iterations"]
) > 3

summary = {
    "convergence_works": convergence_works,
    "best_mechanism": best_mechanism,
    "best_merge": best_merge,
    "semantic_difference": semantic_diff,
}

print("=" * 60)
print("SIMULATION 1: ERGEBNIS-ZUSAMMENFASSUNG")
print("=" * 60)
print()
for k, v in summary.items():
    print(f"  {k}: {v}")
print()

if convergence_works:
    print("ENTSCHEIDUNG: Konvergenz funktioniert. Weiter zu Simulation 2 (Probing).")
    print(f"  Empfohlener Mechanismus: {best_mechanism}")
    print(f"  Empfohlener Merge: {best_merge}")
else:
    print("ENTSCHEIDUNG: Keine Konvergenz. Fundamentalproblem. Ansatz ueberdenken.")

print()
print("Naechste Schritte:")
print("  1. Ergebnisse mit Toby besprechen")
print("  2. Signal-Metriken-Befunde fuer Paper-Draft aufbereiten")
print("  3. Simulation 2 (Probing an bestehendem Modell) planen")